# ViLex-ANND — Đánh giá (T9)
Thứ tự: build index cho 2 model embedding → bật LLM → (sinh nháp câu hỏi → **bạn duyệt tay** → lưu `eval/qa_set.jsonl`)
→ dò τ trên dev → ablation truy xuất trên test → có/không RAG → (tuỳ chọn) FP8 → `make_report`.
Mọi con số trong bài lấy từ `eval/results/`.

In [ ]:
# 1. Mount Drive + vào thư mục dự án
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/vilex-annd'
%cd {PROJECT}
import sys; sys.path.insert(0, PROJECT); sys.path.insert(0, PROJECT + '/scripts')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài thư viện (~3–6 phút). vLLM cài kèm torch tương thích.
!pip install -q -U vllm
!pip install -q -r requirements-colab.txt
import importlib, colab_utils; importlib.reload(colab_utils); from colab_utils import *

In [ ]:
# 3. Chọn LLM. Bản Q4 cộng đồng (Qwen chưa phát hành AWQ chính thức) — kiểm tra model card trước khi dùng.
LLM_REPO    = "QuantTrio/Qwen3.6-35B-A3B-AWQ"        # AWQ 4-bit, nhiều lượt tải nhất
# LLM_REPO  = "palmfuture/Qwen3.6-35B-A3B-GPTQ-Int4" # phương án Q4 khác
# LLM_REPO  = "Qwen/Qwen3.6-35B-A3B-FP8"             # chỉ để đo Q4 vs FP8 trên GPU 80 GB
SERVED_NAME = "qwen3.6-35b-a3b"                      # = llm.model trong config.yaml
MAX_LEN     = 16384
# vLLM được cấp ~30 GiB tuyệt đối (trọng số 23,7 GiB + activation + CUDA graph + cache 16 chuỗi);
# phần còn lại (~5 GiB) cho bge-m3 + reranker của API. A100 40GB -> 0.75, 80GB -> 0.38, 96GB -> 0.31.
GPU_UTIL    = auto_gpu_util()
print("gpu_memory_utilization =", GPU_UTIL, "| cờ tiết kiệm VRAM:", VLLM_LEAN_ARGS)

In [ ]:
# Index cho cả bge-m3 và DEk21 (ablation). Chạy trước khi bật vLLM.
!python -m ingest.run_ingest --strict
for model in ["BAAI/bge-m3", "huyydangg/DEk21_hcmute_embedding"]:
    !EMBED_MODEL={model} python scripts/build_index.py --if-stale
!ls -la index/*

In [ ]:
# 5. Khởi động vLLM chạy nền (lần đầu tải ~25 GB trọng số, 3–10 phút). Log: /content/logs/vllm.log
# Mặc định thêm: --max-num-seqs 16 (mỗi chuỗi giữ ~33 MiB trạng thái Gated DeltaNet; mặc định 256 chuỗi là ~8 GiB),
# --max-num-batched-tokens 4096, và tắt đầu vào ảnh/video (model là bản đa phương thức, không cần cho tra cứu).
# Lỗi thì in thẳng dòng nguyên nhân; chạy xong in bảng VRAM thực tế.
vllm_proc = start_vllm(LLM_REPO, SERVED_NAME, max_len=MAX_LEN, gpu_util=GPU_UTIL)

In [ ]:
# Sinh NHÁP câu hỏi (chỉ khi chưa có qa_set.jsonl). Sau đó tải eval/qa_draft.jsonl về, duyệt/sửa từng câu,
# sửa gold_chunk_ids nếu cần, rồi lưu thành eval/qa_set.jsonl trên Drive.
import os
if not os.path.exists("eval/qa_set.jsonl"):
    !LLM_MODEL={SERVED_NAME} python scripts/gen_qa_draft.py
else:
    print("Đã có eval/qa_set.jsonl")

In [ ]:
# Dò ngưỡng τ trên dev (hybrid + rerank với τ=0) -> sửa retrieval.rerank_threshold trong config.yaml
!python -m eval.run_retrieval --split dev --calibrate --configs hybrid_rerank
!cat eval/results/tau_calibration.csv

In [ ]:
# Ablation truy xuất trên test (6 cấu hình)
!python -m eval.run_retrieval --split test

In [ ]:
# Có RAG vs không RAG (Q4). Xong thì CHẤM TAY cột score (0/1/2) và citation_ok (0/1) trong eval/results/e2e_grading_*.csv
!LLM_MODEL={SERVED_NAME} python -m eval.run_e2e --split test --label q4

In [ ]:
# Tuỳ chọn (GPU 80 GB): Q4 vs FP8 — tắt vLLM Q4, bật bản FP8, chạy lại
RUN_FP8 = False
if RUN_FP8:
    stop(vllm_proc)
    # FP8: trọng số ~35–37 GB (gần gấp rưỡi bản AWQ) -> ngân sách ~44 GiB, chỉ vừa GPU 80 GB
    vllm_proc = start_vllm("Qwen/Qwen3.6-35B-A3B-FP8", SERVED_NAME, max_len=MAX_LEN,
                           gpu_util=auto_gpu_util(budget_gib=44))
    !LLM_MODEL={SERVED_NAME} python -m eval.run_e2e --split test --label fp8

In [ ]:
# Gộp kết quả -> eval/results/report.md + 2 biểu đồ (chạy lại sau khi chấm tay)
!python -m eval.make_report
from IPython.display import Markdown, Image, display
display(Markdown(open("eval/results/report.md", encoding="utf-8").read()))